## Célula 1 — verificar o Python

In [1]:
import sys

print("Versão do Python:", sys.version)

Versão do Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


## Célula 2 — instalar as bibliotecas

In [9]:
%pip install mediapipe==0.10.21 opencv-python numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Found existing installation: mediapipe 0.10.35
Uninstalling mediapipe-0.10.35:
  Successfully uninstalled mediapipe-0.10.35
Note: you may need to restart the kernel to use updated packages.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/51.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/51.0 MB ? eta -:--:--
   - -------------------------------------- 1.6/51.0 MB 5.2 MB/s eta 0:00:10
   --- ------------------------------------ 4.7/51.0 MB 9.5 MB/s eta 0:00:05
   ----- ---------------------------------- 7.3/51.0 MB 10.8 MB/s eta 0:

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Célula 3 — importar as bibliotecas

In [1]:
import time
import math
import cv2
import mediapipe as mp
import numpy as np

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)

OpenCV: 4.11.0
MediaPipe: 0.10.21
NumPy: 1.26.4


# Célula 4 — funções de geometria da mão

In [2]:
def calcular_angulo(ponto_a, ponto_b, ponto_c):
    """
    Calcula o ângulo ABC.

    ponto_a, ponto_b e ponto_c devem possuir:
    - x
    - y
    - z
    """
    vetor_ba = np.array([
        ponto_a.x - ponto_b.x,
        ponto_a.y - ponto_b.y,
        ponto_a.z - ponto_b.z
    ])

    vetor_bc = np.array([
        ponto_c.x - ponto_b.x,
        ponto_c.y - ponto_b.y,
        ponto_c.z - ponto_b.z
    ])

    denominador = np.linalg.norm(vetor_ba) * np.linalg.norm(vetor_bc)

    if denominador == 0:
        return 0.0

    cosseno = np.dot(vetor_ba, vetor_bc) / denominador
    cosseno = np.clip(cosseno, -1.0, 1.0)

    return float(np.degrees(np.arccos(cosseno)))


def dedo_esticado(landmarks, mcp, pip, tip, limite=155):
    """
    Um dedo é considerado esticado quando o ângulo da articulação
    central é aproximadamente uma linha reta.
    """
    angulo = calcular_angulo(
        landmarks[mcp],
        landmarks[pip],
        landmarks[tip]
    )

    return angulo >= limite


def polegar_esticado(landmarks, limite=145):
    """
    Verifica se o polegar está esticado usando sua articulação principal.
    """
    angulo = calcular_angulo(
        landmarks[1],
        landmarks[2],
        landmarks[4]
    )

    return angulo >= limite

# Célula 5 — identificar os dedos levantados

Polegar: 1, 2, 3, 4
Indicador: 5, 6, 7, 8
Médio: 9, 10, 11, 12
Anelar: 13, 14, 15, 16
Mindinho: 17, 18, 19, 20

In [3]:
def analisar_dedos(landmarks):
    dedos = {
        "polegar": polegar_esticado(landmarks),
        "indicador": dedo_esticado(landmarks, 5, 6, 8),
        "medio": dedo_esticado(landmarks, 9, 10, 12),
        "anelar": dedo_esticado(landmarks, 13, 14, 16),
        "mindinho": dedo_esticado(landmarks, 17, 18, 20)
    }

    return dedos


def contar_dedos(dedos):
    return sum(1 for esticado in dedos.values() if esticado)

## Célula 6 — reconhecer números de 1 a 5

Para reduzir falsos positivos, o código considera padrões específicos:

- 1: somente indicador;
- 2: indicador e médio;
- 3: indicador, médio e anelar;
- 4: todos, menos o polegar;
- 5: todos os dedos.

In [4]:
def reconhecer_numero(dedos):
    polegar = dedos["polegar"]
    indicador = dedos["indicador"]
    medio = dedos["medio"]
    anelar = dedos["anelar"]
    mindinho = dedos["mindinho"]

    # Número 1: somente o indicador
    if (
        indicador
        and not polegar
        and not medio
        and not anelar
        and not mindinho
    ):
        return 1

    # Número 2: indicador e médio
    if (
        indicador
        and medio
        and not polegar
        and not anelar
        and not mindinho
    ):
        return 2

    # Número 3: indicador, médio e anelar
    if (
        indicador
        and medio
        and anelar
        and not polegar
        and not mindinho
    ):
        return 3

    # Número 4: quatro dedos, sem o polegar
    if (
        not polegar
        and indicador
        and medio
        and anelar
        and mindinho
    ):
        return 4

    # Número 5: mão completamente aberta
    if (
        polegar
        and indicador
        and medio
        and anelar
        and mindinho
    ):
        return 5

    return None

## Célula 7 — reconhecer o movimento de gatilho

Polegar esticado
Indicador esticado
Médio dobrado
Anelar dobrado
Mindinho dobrado

In [5]:
def gesto_arma_pronta(dedos):
    return (
        dedos["polegar"]
        and dedos["indicador"]
        and not dedos["medio"]
        and not dedos["anelar"]
        and not dedos["mindinho"]
    )


def gesto_gatilho_puxado(dedos):
    return (
        dedos["polegar"]
        and not dedos["indicador"]
        and not dedos["medio"]
        and not dedos["anelar"]
        and not dedos["mindinho"]
    )

# Célula 8 — programa principal

In [7]:
mp_maos = mp.solutions.hands
mp_desenho = mp.solutions.drawing_utils
mp_estilos = mp.solutions.drawing_styles

camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not camera.isOpened():
    raise RuntimeError(
        "Não foi possível abrir a webcam. "
        "Verifique se ela está conectada e se outro programa está usando-a."
    )

# Tenta trabalhar com uma resolução equilibrada.
camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Controle do gatilho
arma_foi_preparada = False
gatilho_bloqueado = False
momento_arma_pronta = 0.0
tempo_maximo_para_puxar = 1.5
intervalo_entre_disparos = 0.45
ultimo_disparo = 0.0

# Controle dos números
numero_candidato = None
numero_exibido = None
inicio_estabilidade = 0.0
tempo_para_confirmar_numero = 0.45

texto_tela = "Mostre a mao direita"
momento_texto = 0.0
duracao_texto = 1.2

try:
    with mp_maos.Hands(
        static_image_mode=False,
        max_num_hands=2,
        model_complexity=1,
        min_detection_confidence=0.65,
        min_tracking_confidence=0.65
    ) as detector_maos:

        while True:
            sucesso, frame = camera.read()

            if not sucesso:
                print("Não foi possível capturar a imagem da webcam.")
                break

            # Espelha a imagem para funcionar como um espelho.
            frame = cv2.flip(frame, 1)

            frame_rgb = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            resultado = detector_maos.process(frame_rgb)

            mao_direita_encontrada = False
            agora = time.time()

            if (
                resultado.multi_hand_landmarks
                and resultado.multi_handedness
            ):
                for landmarks_mao, classificacao in zip(
                    resultado.multi_hand_landmarks,
                    resultado.multi_handedness
                ):
                    lado = classificacao.classification[0].label
                    confianca = classificacao.classification[0].score

                    # Como a imagem foi espelhada antes da detecção,
                    # o MediaPipe deve identificar Right corretamente.
                    if lado != "Right":
                        continue

                    mao_direita_encontrada = True

                    mp_desenho.draw_landmarks(
                        frame,
                        landmarks_mao,
                        mp_maos.HAND_CONNECTIONS,
                        mp_estilos.get_default_hand_landmarks_style(),
                        mp_estilos.get_default_hand_connections_style()
                    )

                    landmarks = landmarks_mao.landmark
                    dedos = analisar_dedos(landmarks)

                    arma_pronta = gesto_arma_pronta(dedos)
                    gatilho_puxado = gesto_gatilho_puxado(dedos)

                    # -----------------------------------------
                    # 1. DETECÇÃO DO GATILHO
                    # -----------------------------------------

                    if arma_pronta:
                        if not arma_foi_preparada:
                            momento_arma_pronta = agora

                        arma_foi_preparada = True
                        gatilho_bloqueado = False

                        texto_tela = "ARMA PRONTA"

                    elif (
                        arma_foi_preparada
                        and gatilho_puxado
                        and not gatilho_bloqueado
                        and agora - momento_arma_pronta <= tempo_maximo_para_puxar
                        and agora - ultimo_disparo >= intervalo_entre_disparos
                    ):
                        print("Puxou o gatilho")

                        texto_tela = "PUXOU O GATILHO"
                        momento_texto = agora

                        ultimo_disparo = agora
                        gatilho_bloqueado = True
                        arma_foi_preparada = False

                    # Cancela a preparação caso demore demais.
                    if (
                        arma_foi_preparada
                        and agora - momento_arma_pronta > tempo_maximo_para_puxar
                    ):
                        arma_foi_preparada = False

                    # -----------------------------------------
                    # 2. DETECÇÃO DOS NÚMEROS
                    # -----------------------------------------

                    numero_atual = reconhecer_numero(dedos)

                    # Não reconhece números enquanto o gesto de arma
                    # estiver sendo realizado.
                    if arma_pronta or gatilho_puxado or arma_foi_preparada:
                        numero_atual = None
                        numero_candidato = None
                        numero_exibido = None

                    if numero_atual is not None:
                        if numero_atual != numero_candidato:
                            numero_candidato = numero_atual
                            inicio_estabilidade = agora

                        elif (
                            agora - inicio_estabilidade
                            >= tempo_para_confirmar_numero
                            and numero_atual != numero_exibido
                        ):
                            print(numero_atual)

                            texto_tela = f"NUMERO {numero_atual}"
                            momento_texto = agora

                            numero_exibido = numero_atual

                    else:
                        numero_candidato = None
                        numero_exibido = None

                    # Informações de depuração.
                    cv2.putText(
                        frame,
                        f"Mao direita: {confianca:.0%}",
                        (20, 35),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.75,
                        (255, 255, 255),
                        2,
                        cv2.LINE_AA
                    )

                    dedos_texto = (
                        f"P:{int(dedos['polegar'])} "
                        f"I:{int(dedos['indicador'])} "
                        f"M:{int(dedos['medio'])} "
                        f"A:{int(dedos['anelar'])} "
                        f"Mi:{int(dedos['mindinho'])}"
                    )

                    cv2.putText(
                        frame,
                        dedos_texto,
                        (20, 70),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.65,
                        (255, 255, 255),
                        2,
                        cv2.LINE_AA
                    )

            if not mao_direita_encontrada:
                texto_tela = "MOSTRE A MAO DIREITA"

                arma_foi_preparada = False
                gatilho_bloqueado = False
                numero_candidato = None
                numero_exibido = None

            # Mantém resultados recentes na tela.
            if agora - momento_texto <= duracao_texto:
                texto_resultado = texto_tela
            elif mao_direita_encontrada:
                texto_resultado = "AGUARDANDO GESTO"
            else:
                texto_resultado = "MOSTRE A MAO DIREITA"

            altura, largura = frame.shape[:2]

            cv2.rectangle(
                frame,
                (0, altura - 90),
                (largura, altura),
                (0, 0, 0),
                -1
            )

            cv2.putText(
                frame,
                texto_resultado,
                (25, altura - 38),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

            cv2.putText(
                frame,
                "Pressione Q para encerrar",
                (largura - 310, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

            cv2.imshow(
                "Controle por gestos",
                frame
            )

            tecla = cv2.waitKey(1) & 0xFF

            if tecla == ord("q"):
                break

finally:
    camera.release()
    cv2.destroyAllWindows()

print("Programa encerrado.")

Puxou o gatilho
Puxou o gatilho
Puxou o gatilho
Puxou o gatilho
Puxou o gatilho
Puxou o gatilho
Puxou o gatilho
5
Programa encerrado.
